In [1]:
import pandas as pd

labeled_df = pd.read_parquet(
    "data/artifacts/02_labeled_orders.parquet"
)

print(labeled_df.shape)
display(labeled_df.head())

(96475, 13)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,total_price,total_freight,num_items,total_payment,is_delayed
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,29.99,8.72,1,38.71,0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,118.70,22.76,1,141.46,0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,159.90,19.22,1,179.12,0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,45.00,27.20,1,72.20,0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,19.90,8.72,1,28.62,0


In [2]:
labeled_df["order_purchase_timestamp"] = pd.to_datetime(
    labeled_df["order_purchase_timestamp"]
)

In [3]:
labeled_df = labeled_df.sort_values(
    "order_purchase_timestamp"
).reset_index(drop=True)

In [4]:
n = len(labeled_df)

train_end = int(n * 0.70)
val_end = int(n * 0.85)

train = labeled_df.iloc[:train_end].copy()
validation = labeled_df.iloc[train_end:val_end].copy()
test = labeled_df.iloc[val_end:].copy()

In [5]:
print("Train:", train.shape)
print("Validation:", validation.shape)
print("Test:", test.shape)

Train: (67532, 13)
Validation: (14471, 13)
Test: (14472, 13)


In [6]:
print("\nTrain dates:")
print(train["order_purchase_timestamp"].min())
print(train["order_purchase_timestamp"].max())

print("\nValidation dates:")
print(validation["order_purchase_timestamp"].min())
print(validation["order_purchase_timestamp"].max())

print("\nTest dates:")
print(test["order_purchase_timestamp"].min())
print(test["order_purchase_timestamp"].max())


Train dates:
2016-10-03 09:44:50
2018-04-15 20:07:56

Validation dates:
2018-04-15 20:10:23
2018-06-21 07:50:39

Test dates:
2018-06-21 08:29:29
2018-08-29 15:00:37


In [7]:
print("Train:")
print(train["is_delayed"].value_counts(normalize=True) * 100)

print("\nValidation:")
print(validation["is_delayed"].value_counts(normalize=True) * 100)

print("\nTest:")
print(test["is_delayed"].value_counts(normalize=True) * 100)

Train:
is_delayed
0    90.973168
1     9.026832
Name: proportion, dtype: float64

Validation:
is_delayed
0    94.658282
1     5.341718
Name: proportion, dtype: float64

Test:
is_delayed
0    93.387231
1     6.612769
Name: proportion, dtype: float64


In [8]:
train.to_parquet(
    "data/artifacts/03_train.parquet",
    index=False
)

validation.to_parquet(
    "data/artifacts/03_validation.parquet",
    index=False
)

test.to_parquet(
    "data/artifacts/03_test.parquet",
    index=False
)

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os


train = pd.read_parquet(
    "data/artifacts/03_train.parquet"
)

print("Shape:", train.shape)
display(train.head())

Shape: (67533, 42)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_lat,customer_lng,...,total_product_volume_cm3,avg_product_photos_qty,avg_product_name_length,avg_product_description_length,seller_customer_distance_km,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,is_delayed
0,bfbd0f9bdef84302105ad712db648a6c,86dc2ffce2dfff336de2f386a786e574,delivered,2016-09-15 12:16:38,2016-09-15 12:16:38,2016-11-07 17:11:53,2016-11-09 07:47:38,2016-10-04,-20.585396,-47.863156,...,12288.0,1.0,34.0,1036.0,566.040211,830d5b7aaa3b6f1e9ad63703bec97d23,14600,sao joaquim da barra,SP,1
1,3b697a20d9e427646d92567910af6d57,355077684019f7f60a031656bd7262b8,delivered,2016-10-03 09:44:50,2016-10-06 15:50:54,2016-10-23 14:02:13,2016-10-26 14:02:13,2016-10-27,-23.581321,-46.635726,...,4096.0,3.0,63.0,1642.0,708.535291,32ea3bdedab835c3aa6cb68ce66565ef,4106,sao paulo,SP,0
2,be5bc2f0da14d8071e2d45451ad119d9,7ec40b22510fdbea1b08921dd39e63d8,delivered,2016-10-03 16:56:50,2016-10-06 16:03:44,2016-10-21 16:33:46,2016-10-27 18:19:38,2016-11-07,-28.291275,-53.501401,...,4096.0,1.0,39.0,518.0,915.734331,2f64e403852e6893ae37485d5fcacdaf,98280,panambi,RS,0
3,65d1e226dfaeb8cdc42f665422522d14,70fc57eeae292675927697fe03ad3ff5,canceled,2016-10-03 21:01:41,2016-10-04 10:18:57,2016-10-25 12:14:28,2016-11-08 10:58:34,2016-11-25,-22.936855,-43.359961,...,3332.0,1.0,25.0,823.0,358.782919,b8b8726af116a5cfb35b0315ecef9172,22770,rio de janeiro,RJ,0
4,a41c8759fbe7aab36ea07e038b2d4465,6f989332712d3222b6571b1cf5b835ce,delivered,2016-10-03 21:13:36,2016-10-05 03:11:49,2016-10-25 11:57:59,2016-11-03 10:58:07,2016-11-29,-30.040958,-51.212970,...,4160.0,1.0,39.0,141.0,818.048765,61db744d2f835035a5625b59350c6b63,90040,porto alegre,RS,0
